# Speech Recognition — Colab Training

Train the LSTM-CTC model on **Google Colab GPU**.

**Before running:** Runtime → Change runtime type → **T4 GPU**

**Important:** Mount Google Drive **before** training so models save to Drive every epoch (survives disconnect).

**Steps:**
1. Clone repo
2. Install dependencies
3. Download LibriSpeech (~6 GB) — or skip if already on Drive
4. **Mount Google Drive** (required)
5. Train — saves directly to Drive after each epoch
6. Download `.pth` to your PC for `app.py`

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU. Enable Runtime → Change runtime type → T4 GPU')

In [ ]:
# Clone the project (replace with your fork URL if needed)
!git clone https://github.com/sumathis15/Speech_Recognition.git
%cd Speech_Recognition

In [ ]:
!pip install -q -r requirements.txt

## Download LibriSpeech (train-clean-100)
Downloads ~6 GB from OpenSLR. Takes 10–20 minutes depending on connection.

In [ ]:
import os
import tarfile

DATA_DIR = 'data/raw/LibriSpeech/train-clean-100'
ARCHIVE = 'data/raw/train-clean-100.tar.gz'
URL = 'http://www.openslr.org/resources/12/train-clean-100.tar.gz'
MIN_SIZE = 6_000_000_000  # full file is ~6.3 GB

os.makedirs('data/raw', exist_ok=True)

def count_flacs():
    if not os.path.isdir(DATA_DIR):
        return 0
    return sum(1 for r, _, files in os.walk(DATA_DIR) for f in files if f.endswith('.flac'))

existing = count_flacs()
if existing >= 28000:
    print(f'Dataset already present ({existing} FLAC files). Skipping download.')
else:
    if os.path.exists(ARCHIVE) and os.path.getsize(ARCHIVE) < MIN_SIZE:
        bad_size = os.path.getsize(ARCHIVE)
        print(f'Removing broken partial download ({bad_size:,} bytes)...')
        os.remove(ARCHIVE)

    if not os.path.exists(ARCHIVE) or os.path.getsize(ARCHIVE) < MIN_SIZE:
        print('Downloading train-clean-100 (~6.3 GB). Expect 15-40 minutes...')
        print('Do not stop this cell until the download finishes.')
        !wget -c --tries=5 --timeout=60 --read-timeout=60 --show-progress -O {ARCHIVE} {URL}

    size = os.path.getsize(ARCHIVE)
    print(f'Archive size: {size / 1e9:.2f} GB')
    if size < MIN_SIZE:
        raise RuntimeError(
            'Download incomplete. Re-run this cell to resume (wget -c continues). '
            'If it keeps failing, download manually from https://www.openslr.org/12/ '
            'and upload to Colab Files, then extract with: '
            '!tar -xzf /content/train-clean-100.tar.gz -C data/raw/'
        )

    if not tarfile.is_tarfile(ARCHIVE):
        os.remove(ARCHIVE)
        raise RuntimeError('Downloaded file is not a valid archive (often an error page). Re-run to retry.')

    print('Extracting (5-10 minutes)...')
    !tar -xzf {ARCHIVE} -C data/raw/

    if count_flacs() < 28000:
        raise RuntimeError('Extraction finished but FLAC count looks wrong. Check data/raw/LibriSpeech/')

    print('Removing archive to free disk space...')
    !rm -f {ARCHIVE}
    print('Download and extract complete.')

flac_count = count_flacs()
print(f'FLAC files found: {flac_count}')
assert flac_count >= 28000, 'Expected ~28539 FLAC files'

## Mount Google Drive (REQUIRED — do this before training)
Models save to Drive after every epoch so you keep progress if Colab disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_MODEL_DIR = '/content/drive/MyDrive/Speech_Recognition/model'
os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)
print('Model will be copied to:', DRIVE_MODEL_DIR)

## Train on full dataset
Saves **directly to Google Drive** every epoch (~1–2 sec overhead per epoch, negligible).
Expect ~15–30 min per epoch on T4 GPU.

In [ ]:
# Batch size: try 64 on T4 GPU. If you get CUDA out-of-memory, use 32 or 48.
BATCH_SIZE = 64

DRIVE_MODEL = '/content/drive/MyDrive/Speech_Recognition/model'
DRIVE_CKPT = f'{DRIVE_MODEL}/checkpoints'
!python train.py \
  --epochs 30 \
  --batch-size {BATCH_SIZE} \
  --num-workers 2 \
  --best-model-path {DRIVE_MODEL}/lstm_ctc_model_best.pth \
  --model-path {DRIVE_MODEL}/lstm_ctc_model.pth \
  --checkpoint-dir {DRIVE_CKPT}

## Resume training (if Colab disconnected)
Run this **instead of** the train cell above. Uses checkpoints on Google Drive.

In [ ]:
BATCH_SIZE = 64
DRIVE_MODEL = '/content/drive/MyDrive/Speech_Recognition/model'
DRIVE_CKPT = f'{DRIVE_MODEL}/checkpoints'

RESUME_ARGS = '--resume'
if not os.path.exists(f'{DRIVE_CKPT}/training_state.pt'):
    RESUME_ARGS += ' --start-epoch 22 --initial-best-cer 0.1387'

!python train.py \
  --epochs 30 \
  --batch-size {BATCH_SIZE} \
  --num-workers 2 \
  --best-model-path {DRIVE_MODEL}/lstm_ctc_model_best.pth \
  --model-path {DRIVE_MODEL}/lstm_ctc_model.pth \
  --checkpoint-dir {DRIVE_CKPT} \
  {RESUME_ARGS}

## Download model to your PC
Files are already on Google Drive. This downloads the best model for local `app.py`.

In [ ]:
from google.colab import files

model_on_drive = f'{DRIVE_MODEL}/lstm_ctc_model.pth'
best_on_drive = f'{DRIVE_MODEL}/lstm_ctc_model_best.pth'

if os.path.exists(best_on_drive):
    print('Saved on Google Drive:')
    print(' ', best_on_drive)
    print(' ', model_on_drive)
    files.download(best_on_drive)
    print('Download started — put file in C:\\Speech_Recognition\\model\\lstm_ctc_model.pth')
else:
    print('Model not found on Drive. Run Drive mount + training cells first.')